# Cross-Evaluation: Train-time vs Inference-time Preprocessing

**Goal:** Compare Raw and v1/v2/v3 *independently*, without feeding v2 inputs to a v1 model (or other policy-to-policy combinations). Each policy forms a fixed 2×2 design: A = Raw/Raw, B = Raw/policy, C = policy/Raw, D = policy/policy.

**Inputs:** Frozen five-class development images and labels; versioned run checkpoints, metrics and per-image matching statistics. v3 may still be training.

**Outputs:** Read-only tables/figures; if and only if `ALLOW_INFERENCE=True`, missing evaluations are saved under `runs/comparisons/` (diagonals remain owned by their `04_eval` notebooks). Optional CPU policy timings are in memory.

**Prerequisites:** Completed data export and training for each included model; development-only access. Final holdout stays sealed. A missing v3 is reported as blocked, without blocking v0–v2.

**Outline**

- [1. Configuration](#section-1)
- [2. Evaluation Matrix & Preflight](#section-2)
- [3. Run / Reuse Evaluation](#section-3)
- [4. Accuracy & Factorial Contrasts](#section-4)
- [5. Per-Class & Cohort Analysis](#section-5)
- [6. Cost & Efficiency](#section-6)
- [7. Summary](#section-7)

<a id="section-1"></a>

## 1. Configuration

Run All is safe by default. `RUN_DATES=None` selects the newest run (including a currently training v3, which is then shown as blocked). Pin explicit dates for reproducible comparisons. Setting `BOOTSTRAP_RESAMPLES=500` enables paired, source-group 95% CIs; 1,000+ draws are preferable for a final exploratory report. CIs are conditional on the fixed single-seed checkpoints.

In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from vehicle_detection.common import project_root, require_paths
from vehicle_detection.comparison import (
    POLICIES,
    benchmark_preprocessing,
    compare_policies,
    evaluate_matrix,
    inspect_matrix,
    policy_cohorts,
    result_table,
    training_cost,
)
from vehicle_detection.data import FIVE_CLASSES

ROOT = project_root()
os.chdir(ROOT)
require_paths(runs=ROOT / "runs", split=ROOT / "configs/data.yaml")

RUN_DATES = {"v0": None, "v1": None, "v2": None, "v3": None}
ALLOW_INFERENCE = False  # Explicit opt-in for missing cells; NEVER starts training
BOOTSTRAP_RESAMPLES = 0  # Set 500 or 1000 for paired 95% CIs
BOOTSTRAP_SEED = 2026
MEASURE_PREPROCESS_COST = False
COST_SAMPLES = 128
COST_REPEATS = 3

<a id="section-2"></a>

## 2. Evaluation Matrix & Preflight

Exactly 10 permitted pairs: Raw model sees all four inputs; each processed model sees only Raw and its own input. Valid cached results are reused. Missing/incomplete v3 is marked **blocked**, while invalid cache identities are **stale** and must be inspected manually. The preflight verifies frozen exports once per run, not once per cell.

In [ ]:
report = inspect_matrix(ROOT, run_dates=RUN_DATES)
display(pd.DataFrame(report["rows"]))
if report["blocked"]:
    print("Not ready:", report["blocked"])
print("Ready models:", sorted(report["contexts"]))
print("Saved cells:", len(report["results"]), "/ 10")

<a id="section-3"></a>

## 3. Run / Reuse Evaluation

No inference occurs unless `ALLOW_INFERENCE=True`. Missing diagonals use the existing version-owned `04_eval` implementation; new cross-cells use native FP32 `YOLO.val` with the *other* run's already exported images and the selected model's `best.pt`. Incomplete/stale directories are never overwritten automatically.

In [ ]:
report = evaluate_matrix(
    ROOT,
    run_dates=RUN_DATES,
    allow_inference=ALLOW_INFERENCE,
    preflight=report,
)
overview = result_table(report)
display(overview)
print("Cached or newly evaluated:", overview.status.isin(["cached", "evaluated"]).sum())
print("Pending:", overview.status.value_counts().to_dict())

<a id="section-4"></a>

## 4. Accuracy & Factorial Contrasts

All mAP and delta values shown here are percentages and **percentage points** respectively. For each policy: **B−A** = preprocessing effect on Raw model; **D−B** = matched-training effect at processed input; **C−A** = training effect on Raw input; **D−C** = preprocessing effect on policy-trained model; **D−A** = whole-pipeline change; **interaction** = (D−C)−(B−A).

With `BOOTSTRAP_RESAMPLES > 1`, one source-group resample is shared across all four arms of each 2×2. Intervals do not include variation from additional training seeds, checkpoint selection, or multiple-comparison adjustment.

In [ ]:
contrast_df, intervals = compare_policies(
    report, bootstrap_resamples=BOOTSTRAP_RESAMPLES, seed=BOOTSTRAP_SEED
)
if contrast_df.empty:
    print("No complete 2x2 policy yet; available diagonal/cross rows:")
    display(overview[["model", "input", "status", "mAP50_95_pct"]].round(3))
else:
    display(contrast_df.round(3))
    effects = contrast_df.pivot(index="policy", columns="contrast", values="delta_pp")
    ax = effects[
        ["inference_on_raw_model", "training_on_processed_input", "joint_pipeline"]
    ].plot.bar(figsize=(11, 4), rot=0)
    ax.set(
        xlabel="Policy",
        ylabel="Delta mAP50:95 (pp)",
        title="Preprocessing vs matched training vs complete pipeline",
    )
    ax.axhline(0, linewidth=1)
    plt.tight_layout()
    plt.show()

for policy in POLICIES:
    keys = [("v0", "v0"), ("v0", policy), (policy, "v0"), (policy, policy)]
    if not all(f"{m}/{i}" in report["results"] for m, i in keys):
        continue
    display(
        pd.DataFrame(
            [
                {
                    "model": m,
                    "input": i,
                    "mAP50:95 (%)": report["results"][f"{m}/{i}"]["overall"]["mAP50_95"] * 100,
                    "mAP50 (%)": report["results"][f"{m}/{i}"]["overall"]["mAP50"] * 100,
                }
                for m, i in keys
            ]
        )
        .set_index(["model", "input"])
        .round(3)
    )
    if policy in intervals:
        print(
            policy,
            "source groups:",
            intervals[policy]["groups"],
            "| valid bootstrap draws:",
            {name: v["valid_draws"] for name, v in intervals[policy]["contrasts"].items()},
        )

<a id="section-5"></a>

## 5. Per-Class & Cohort Analysis

Compare all five vehicle classes. Cohorts use **raw-image** brightness, and policy-defined Apply/Skip IDs stay fixed for every arm in that policy's 2×2. This prevents comparing different populations. v2 processes every image; an absent Skip cohort is **not** an AP of zero.

In [ ]:
class_rows = []
for key, result in report["results"].items():
    model, inputs = key.split("/")
    for item in result["classes"]:
        class_rows.append(
            {
                "model": model,
                "input": inputs,
                "class": item["Class"],
                "mAP50 (%)": item["mAP50"] * 100,
                "mAP50:95 (%)": item["mAP50-95"] * 100,
            }
        )
if class_rows:
    class_frame = pd.DataFrame(class_rows)
    display(
        class_frame.pivot_table(index="class", columns=["model", "input"], values="mAP50:95 (%)")
        .reindex(FIVE_CLASSES)
        .round(3)
    )
else:
    print("No evaluated cells yet.")


# Optional class-level 95% CIs share the same group draws as the overall contrasts.
class_ci_rows = []
for policy, summary in intervals.items():
    for class_id, estimates in summary["per_class"].items():
        for contrast in (
            "inference_on_raw_model",
            "training_on_processed_input",
            "joint_pipeline",
        ):
            item = estimates[contrast]
            ci = item["ci95"]
            class_ci_rows.append(
                {
                    "policy": policy,
                    "class": FIVE_CLASSES[class_id],
                    "contrast": contrast,
                    "delta_pp": item["delta"] * 100,
                    "ci_low_pp": ci[0] * 100 if ci else float("nan"),
                    "ci_high_pp": ci[1] * 100 if ci else float("nan"),
                    "valid_draws": item["valid_draws"],
                }
            )
if class_ci_rows:
    print("Per-class paired 95% CIs (unadjusted, conditional on fixed checkpoints)")
    display(pd.DataFrame(class_ci_rows).round(3))

for policy in POLICIES:
    if policy not in report["contexts"]:
        continue
    print("Cohorts:", policy)
    cohort_frame = policy_cohorts(report, policy=policy)
    if cohort_frame.empty:
        print("No available cells")
    else:
        display(
            cohort_frame.pivot_table(
                index=["cohort", "images"], columns=["model", "input"], values="mAP50_95_pct"
            ).round(3)
        )

<a id="section-6"></a>

## 6. Cost & Efficiency

Native Ultralytics `speed_ms` excludes the external v1/v2/v3 routing/operator cost; it is **not end-to-end inference latency**. Training time is reported from saved `results.csv`. Optional CPU policy measurements use decoded **Raw JPEG95 exports as proxy images**, with one OpenCV thread; they exclude decode/encode/disk/GPU costs and do not recreate the exact original-source export. Offline preprocessing is paid once; online preprocessing adds runtime latency.

In [ ]:
display(training_cost(report).round(3))
speed_rows = []
for key, result in report["results"].items():
    speed = result.get("speed_ms", {})
    speed_rows.append(
        {
            "pair": key,
            "native_preprocess_ms": speed.get("preprocess"),
            "native_inference_ms": speed.get("inference"),
            "native_postprocess_ms": speed.get("postprocess"),
        }
    )
if speed_rows:
    display(pd.DataFrame(speed_rows).set_index("pair").round(3))
if MEASURE_PREPROCESS_COST:
    cpu_cost = benchmark_preprocessing(
        report, samples=COST_SAMPLES, repeats=COST_REPEATS, seed=BOOTSTRAP_SEED
    )
    display(cpu_cost.round(3))
    print("CPU policy timing is not an end-to-end streaming latency benchmark.")
else:
    print("Set MEASURE_PREPROCESS_COST=True to benchmark CPU routing/operators.")

<a id="section-7"></a>

## 7. Summary

Interpret **B−A** first: can a Raw-trained model benefit from the policy only at inference? Then **D−B**: is training on processed images worth the additional training/data-preparation effort? Finally inspect **D−A** and class/cohort regressions. A confidence interval crossing zero is inconclusive, not proof of equivalence. This is single-seed development evidence; **the final holdout remains sealed**.

In [ ]:
ready = overview[overview.status.isin(["cached", "evaluated"])]
print(f"Completed/reused: {len(ready)}/10 | pending: {10 - len(ready)}")
if not contrast_df.empty:
    for policy in POLICIES:
        subset = contrast_df.loc[
            (contrast_df.policy == policy)
            & (
                contrast_df.contrast.isin(
                    [
                        "inference_on_raw_model",
                        "training_on_processed_input",
                        "joint_pipeline",
                    ]
                )
            )
        ]
        if not subset.empty:
            print("\n" + policy + " (mAP50:95 delta in percentage points):")
            display(subset.set_index("contrast").drop(columns="policy").round(3))
print("\nDevelopment only; final holdout not accessed. No training launched.")